In [11]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
print(f"In colab: {IN_COLAB}")

In colab: True


In [12]:
if IN_COLAB:
  from pathlib import Path
  from google.colab import drive
  from datetime import datetime

  ROOT_PATH = Path('/content/drive/MyDrive/')
  OUTPUT_ROOT_PATH = ROOT_PATH / f"Colab_Data/gsm-symbolic/outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"

  drive.mount(str(ROOT_PATH.parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
if IN_COLAB:
  import os

  DRIVE_PROJECT_PATH = ROOT_PATH / "Colab_Projects"
  REPO_FOLDER_NAME = 'gsm-symbolic-benchmarking'
  FULL_REPO_PATH = DRIVE_PROJECT_PATH / REPO_FOLDER_NAME

  import os
  os.chdir(FULL_REPO_PATH)
  !git pull origin main

  try:
    import gsm_benchmarker
  except ImportError:
    !pip install transformers datasets accelerate bitsandbytes numpy pandas tqdm psutil torch cloud-tpu-client anthropic openai google-genai  # other stuff probably missing too
    !pip install -e .
    os.kill(os.getpid(), 9)  # need to restart the session for import to be possible

  os.chdir("..")

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 14 (delta 10), reused 14 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 1.40 KiB | 4.00 KiB/s, done.
From https://github.com/the-mysh/gsm-symbolic-benchmarking
 * branch            main       -> FETCH_HEAD
   1b94b7f..845a48d  main       -> origin/main
Updating 1b94b7f..845a48d
Fast-forward
 src/gsm_benchmarker/benchmark.py       | 12 +++++++++---
 src/gsm_benchmarker/model_evaluator.py | 20 +++++---------------
 src/gsm_benchmarker/utils/path_ops.py  |  8 ++++++++
 3 files changed, 22 insertions(+), 18 deletions(-)


In [14]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner, APIType
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger

if IN_COLAB:
  from google.colab import userdata
  hf_api_token = userdata.get('HF_TOKEN')
  os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
else:
  hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(hf_api_token)

whoami()['name']


'the-mysh'

In [15]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [16]:
models_config = ModelsConfig()
all_models = models_config.all_models
print(models_config.all_models)  # note: openai - insufficient quota for benchmarking

[('google/gemma-2b', None), ('google/gemma-2b-it', None), ('google/gemma-7b', None), ('google/gemma-7b-it', None), ('google/gemma-2-2b', None), ('google/gemma-2-2b-it', None), ('google/gemma-2-9b', None), ('google/gemma-2-9b-it', None), ('google/gemma-2-27b-it', None), ('microsoft/phi-2', None), ('microsoft/Phi-3-mini-128k-instruct', None), ('microsoft/Phi-3-small-128k-instruct', None), ('microsoft/Phi-3-medium-128k-instruct', None), ('microsoft/Phi-3.5-mini-instruct', None), ('mistralai/Mistral-7B-v0.1', None), ('mistralai/Mistral-7B-Instruct-v0.1', None), ('mistralai/Mistral-7B-v0.3', None), ('mistralai/Mistral-7B-Instruct-v0.3', None), ('mistralai/Mathstral-7B-v0.1', None), ('meta-llama/Meta-Llama-3-8B', None), ('meta-llama/Meta-Llama-3-8B-Instruct', None), ('gpt-4o-mini', <APIType.openai: 1>), ('gpt-4o', <APIType.openai: 1>), ('o1-mini', <APIType.openai: 1>), ('o1-preview', <APIType.openai: 1>)]


In [17]:
open_models = models_config.open_models
print(open_models)

['google/gemma-2b', 'google/gemma-2b-it', 'google/gemma-7b', 'google/gemma-7b-it', 'google/gemma-2-2b', 'google/gemma-2-2b-it', 'google/gemma-2-9b', 'google/gemma-2-9b-it', 'google/gemma-2-27b-it', 'microsoft/phi-2', 'microsoft/Phi-3-mini-128k-instruct', 'microsoft/Phi-3-small-128k-instruct', 'microsoft/Phi-3-medium-128k-instruct', 'microsoft/Phi-3.5-mini-instruct', 'mistralai/Mistral-7B-v0.1', 'mistralai/Mistral-7B-Instruct-v0.1', 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistralai/Mathstral-7B-v0.1', 'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct']


In [18]:
# these are not in the paper, but interestingly give very good results (at least on a small subset of the benchmark data, main variant)
# bigger quota than for openAI models, but likely still too small for full benchmark
other_closed_models = [
    ("gemini-2.0-flash", APIType.google_genai),
    ("gemini-2.5-flash", APIType.google_genai),
]

In [19]:
variants = GSMSymbolicDataset.Variant

br = BenchmarkRunner(
    models=open_models,
    dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
    storage_path=OUTPUT_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True, gpu0_max_memory="14GB", cpu_max_memory="10GB")
)

gsm_benchmarker.utils.path_ops [INFO] 2025-10-17 16:35:46: Creating directories at /content/drive/MyDrive/Colab_Data/gsm-symbolic/outputs/20251017_163539


In [20]:
br.run(n_sets=2, n_per_set=2)

gsm_benchmarker.benchmark [INFO] 2025-10-17 16:35:46: ========== Evaluating model 1/21: google/gemma-2b ==========
gsm_benchmarker.model_evaluator [DEBUG] 2025-10-17 16:35:46: Initialising a HuggingFace model
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 16:35:46: Setting up model google/gemma-2b
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-17 16:35:46: Loading tokeniser
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 16:35:48: CUDA available
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-17 16:35:48: Loading model with CUDA


KeyboardInterrupt: 

In [ ]:
br.summarise_results()

In [ ]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[0]].loc[0, 0]
el

In [ ]:
print(el.question)
print()
print(el.response)